# 13.4.2 The MUSCL–Hancock Method (MHM)

<img src="images/image.png" width="600">

# 14.4.2 A Variant of the Scheme

<img src="images/image copy.png" width="600">

In [ ]:
def muscl_reconstruction(self, CELL, dt, normal='x'):
    """
    Perform MUSCL-Hancock-type reconstruction (without TVD limiter).

    Parameters
    ----------
    CELL : torch.Tensor
        Cell-averaged primitive variables, shape (Nz+2, Ny+2, Nx+2, 5)
        [rho, u, v, w, p]

    dt : float
        Time step

    normal : str
        'x' or 'y' or 'z'

    Returns
    -------
    WL, WR : torch.Tensor
        Reconstructed left/right states at interfaces along `normal`.

        For normal='x': shape (Nz+2, Ny+2, Nx, 5)
        For normal='y': shape (Nz+2, Ny, Nx+2, 5)
        For normal='z': shape (Nz, Ny+2, Nx+2, 5)

    Wil' = Wi - 0.5 * Di - 0.5 * dt / dx * A Di
    WiR' = Wi + 0.5 * Di - 0.5 * dt / dx * A Di
    
    """
    # -------- pick normal velocity component u_n --------
    if normal == 'x':
        Di = 0.5 * (CELL[:, :, 2:, :] - CELL[:, :, :-2, :])
        Wi = CELL[:, :, 1:-1, :]  
        dx = self.dx  
    elif normal == 'y':
        Di = 0.5 * (CELL[:, 2:, :, :] - CELL[:, :-2, :, :])
        Wi = CELL[:, 1:-1, :, :]
        dx = self.dy
    elif normal == 'z':
        Di = 0.5 * (CELL[2:, :, :, :] - CELL[:-2, :, :, :])
        Wi = CELL[1:-1, :, :, :]
        dx = self.dz
    else:
        raise ValueError("normal must be 'x' or 'y' or 'z'")

    #calculate (A(Wi) + B(Wi) + C(Wi)) Di = K
    rho = Wi[..., 0]
    u = Wi[..., 1]
    v = Wi[..., 2]
    w = Wi[..., 3]
    p = Wi[..., 4]

    rho_x = Di[..., 0]
    u_x = Di[..., 1]
    v_x = Di[..., 2]
    w_x = Di[..., 3]
    p_x = Di[..., 4]

    K = torch.zeros_like(Wi)

    if(normal == 'x'):
        K[..., 0] = u * rho_x + rho * u_x
        K[..., 1] = u * u_x + p_x / rho
        K[..., 2] = u * v_x
        K[..., 3] = u * w_x
        K[..., 4] = u * p_x + self.GAMMA * p * u_x
    elif(normal == 'y'):
        K[..., 0] = v * rho_x + rho * v_x
        K[..., 1] = v * u_x
        K[..., 2] = v * v_x + p_x / rho
        K[..., 3] = v * w_x
        K[..., 4] = v * p_x + self.GAMMA * p * v_x
    elif(normal == 'z'):
        K[..., 0] = w * rho_x + rho * w_x
        K[..., 1] = w * u_x
        K[..., 2] = w * v_x
        K[..., 3] = w * w_x + p_x / rho
        K[..., 4] = w * p_x + self.GAMMA * p * w_x
    else:
        raise ValueError("normal must be 'x' or 'y' or 'z'")

    WL = Wi - 0.5 * Di - 0.5 * dt / dx * K
    WR = Wi + 0.5 * Di - 0.5 * dt / dx * K

    # WL and Wr are the left and right states at the cell.
    # the function is returning the left and right states at the interface between the cell and the next cell.

    if(normal == 'x'):
        return WR[:, :, :-1, :], WL[:, :, 1:, :]
    elif(normal == 'y'):
        return WR[:, :-1, :, :], WL[:, 1:, :, :]
    elif(normal == 'z'):
        return WR[:-1, :, :, :], WL[1:, :, :, :]
    else:
        raise ValueError("normal must be 'x' or 'y' or 'z'")